# Block 0. Cài đặt thư viện và Kết nối Google Drive

In [ ]:
# Cài đặt XGBoost
!pip install -q xgboost

# Kết nối Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Nạp thư viện
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_log_error
from xgboost import XGBRegressor

Mounted at /content/drive


# Block 1. Cấu hình Đường dẫn và Tải Dữ liệu

In [ ]:
# Cấu hình đường dẫn
DATA_DIR = Path("/content/drive/MyDrive/insurance premium data")
TRAIN_FILE = DATA_DIR / "train.csv"
TEST_FILE = DATA_DIR / "test.csv"
SUBMISSION_FILE = DATA_DIR / "submission_final.csv"

TARGET = "Premium Amount"
ID_COL = "id"
SEED = 42

# Tải dữ liệu
train_df = pd.read_csv(TRAIN_FILE)
test_df = pd.read_csv(TEST_FILE)

# Xác định các cột dùng để dự đoán
FEATURES = [col for col in train_df.columns if col not in [ID_COL, TARGET]]

# Tách biến cho cho dữ liệu gốc
X_full = train_df[FEATURES].copy()
y_full = train_df[TARGET].copy()

# Tách biến cho tập Test
X_test = test_df[FEATURES].copy()

# Block 2. Kỹ thuật Trích xuất Đặc trưng (Feature Engineering)

In [ ]:
def create_features(X_data):
    X_data = X_data.copy()
    if "Policy Start Date" in X_data.columns:
        date = pd.to_datetime(X_data["Policy Start Date"], errors="coerce")
        X_data["Policy Start Year"] = date.dt.year
        X_data["Policy Start Month"] = date.dt.month
        X_data["Policy Start Day"] = date.dt.day
        X_data["Policy Start DayOfWeek"] = date.dt.dayofweek

        X_data = X_data.drop(columns=["Policy Start Date"])
    return X_data

# Áp dụng cho cả tập Full Train và tập Test
X_full = create_features(X_full)
X_test = create_features(X_test)

# Block 3. Xây dựng bộ tự động tiền xử lý (Preprocessor)

In [ ]:
# Phân loại cột chữ và số
cat_features = X_full.select_dtypes(include=["object", "category"]).columns.tolist()
num_features = [col for col in X_full.columns if col not in cat_features]

# Khối xử lý cho biến số
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# Khối xử lý cho biến chữ
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Gộp thành bộ điều phối tổng
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, num_features),
    ("cat", categorical_pipeline, cat_features)
])

# Block 4. Validation

In [ ]:
# Chia tầng phân phối giá tiền
target_bins = pd.qcut(y_full, q=20, labels=False, duplicates="drop")

# Cắt 80% Train và 20% Validation
X_train, X_val, y_train, y_val = train_test_split(
    X_full, y_full,
    test_size=0.20,
    random_state=SEED,
    stratify=target_bins
)

# Cấu hình mô hình thử nghiệm
model_val = XGBRegressor(
    n_estimators=500,
    max_depth=6,         # Thử thay đổi số này (vd: 4, 8, 10)
    learning_rate=0.05,  # Thử thay đổi số này (vd: 0.01, 0.1)
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=SEED,
    n_jobs=-1
)

# Huấn luyện trên tập 80%
pipeline_val = Pipeline([("preprocessor", preprocessor), ("model", model_val)])
pipeline_val.fit(X_train, y_train)

# Chấm điểm trên tập 20%
val_pred = pipeline_val.predict(X_val)
val_pred = np.clip(val_pred, 0, None)
rmsle = np.sqrt(mean_squared_log_error(y_val, val_pred))

print(f" ĐIỂM RMSLE TRÊN TẬP VALIDATION: {rmsle:.6f}")

 ĐIỂM RMSLE TRÊN TẬP VALIDATION: 1.137593


# Block 5. Full Training

In [ ]:
# Dùng lại cấu hình tốt nhất sau khi thử
final_model = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=SEED,
    n_jobs=-1
)

final_pipeline = Pipeline([("preprocessor", preprocessor), ("model", final_model)])

# Huấn luyện trên toàn bộ X_full và y_full (Không dùng X_train nữa)
final_pipeline.fit(X_full, y_full)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Age', 'Annual Income',
                                                   'Number of Dependents',
                                                   'Health Score',
                                                   'Previous Claims',
                                                   'Vehicle Age',
                                                   'Credit Score',
                                                   'Insurance Duration',
                                                   'Policy Start Year',
                                                   'Policy Start Month',
                                                   'Policy Start Day',
                                                   'Policy Start DayOfWeek']),
                                                 ('cat',
                                                  Pipeline(...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.05,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=6, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=500, n_jobs=-1,
                              num_parallel_tree=None, ...))])

# Block 6. DỰ ĐOÁN & XUẤT FILE

In [ ]:
# Dự đoán
test_pred = final_pipeline.predict(X_test)
test_pred = np.clip(test_pred, 0, None)

# Gom kết quả thành bảng
submission = pd.DataFrame({
    ID_COL: test_df[ID_COL],
    TARGET: test_pred
})

# Lưu file CSV
submission.to_csv(SUBMISSION_FILE, index=False)